In [ ]:
import pickle
import numpy as np
from scipy.stats import spearmanr

PKL_PATH = './best_pkl/stfo_wo_spin/retrained_best_stfo_wo_spin_ising_lite.pkl'

with open(PKL_PATH, 'rb') as f:
    d = pickle.load(f)

print('Keys:', list(d.keys()))

In [ ]:
preds   = d['preds_all']
targets = d['targets_all']
comps   = d['comps_all']
splits  = d['splits_all']   # 0=train, 1=val, 2=test

n_atoms_orig = d.get('n_atoms_orig', d.get('n_atoms', 1))

split_label = {0: 'train', 1: 'val', 2: 'test'}

print(f'Total samples: {len(preds)}')
for s, name in split_label.items():
    print(f'  {name}: {(splits == s).sum()}')

In [ ]:
print('Per-composition SRCC (all splits combined)\n')
print(f'{"comp":>8}  {"N":>5}  {"SRCC":>7}  {"RMSE(eV/at)":>12}  train/val/test')
print('-' * 60)

for c in sorted(set(comps)):
    mask = comps == c
    n = mask.sum()
    if n < 3:
        continue
    rho  = spearmanr(targets[mask], preds[mask]).correlation
    rmse = np.sqrt(np.mean((targets[mask] - preds[mask])**2)) / n_atoms_orig

    split_counts = '/'.join(str((splits[mask] == s).sum()) for s in [0, 1, 2])
    print(f'x={c/1000:.3f}  {n:>5}  {rho:>7.4f}  {rmse:>12.6f}  {split_counts}')